# Huấn luyện mô hình nhận diện đồ nội thất

**Cách dùng:**

1. Bật GPU: **Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU → Lưu**
2. Chạy **từng ô một** từ trên xuống (bấm nút ▶ ở đầu mỗi ô), chờ ô trước xong hẳn rồi mới chạy ô sau.
3. Ô nào hiện chữ đỏ thì dừng lại, chụp màn hình gửi đi — đừng chạy tiếp.

Tổng thời gian khoảng 1,5 – 3 tiếng. **Để tab này mở**, không Colab tự ngắt.


## 1. Kiểm tra GPU

Phải in ra tên GPU, ví dụ `Tesla T4`.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit('CHUA BAT GPU! Vao: Thoi gian chay > Thay doi loai thoi gian chay > T4 GPU > Luu, roi chay lai o nay.')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Cài thư viện

Khoảng 2–4 phút. Cuối ô phải in ra `DA CAI XONG`.

In [ ]:
!pip install -q ultralytics fiftyone boto3

import ultralytics, fiftyone, boto3
print('ultralytics', ultralytics.__version__)
print('fiftyone  ', fiftyone.__version__)
print('DA CAI XONG')


## 3. Tải ảnh nội thất đã có sẵn nhãn

Lấy từ bộ **Open Images V7** của Google — ảnh đã gán nhãn sẵn, không phải ngồi khoanh tay.

Khoảng 20–40 phút, tải 3–6 GB. Cuối ô phải in ra `DA CHUAN BI XONG DU LIEU`.

Muốn mô hình giỏi hơn thì tăng `SO_ANH_TRAIN` lên 12000–15000, đổi lại lâu hơn.

In [ ]:
import os, fiftyone as fo, fiftyone.zoo as foz

SO_ANH_TRAIN = 6000
SO_ANH_VAL   = 1200

LOP = ['Table', 'Desk', 'Coffee table', 'Kitchen & dining room table',
       'Chair', 'Stool', 'Bench', 'Couch', 'Bed',
       'Wardrobe', 'Cabinetry', 'Cupboard', 'Chest of drawers',
       'Nightstand', 'Drawer', 'Shelf', 'Bookcase', 'Filing cabinet']

THU_MUC = '/content/noi_that'

train = foz.load_zoo_dataset('open-images-v7', split='train',
                             label_types=['detections'], classes=LOP,
                             max_samples=SO_ANH_TRAIN, dataset_name='nt_train')
val = foz.load_zoo_dataset('open-images-v7', split='validation',
                           label_types=['detections'], classes=LOP,
                           max_samples=SO_ANH_VAL, dataset_name='nt_val')

# Tim dung ten truong chua nhan (tuy phien ban fiftyone ma ten khac nhau)
def truong_nhan(bo):
    for ten in ('detections', 'ground_truth', 'positive_labels'):
        if ten in bo.get_field_schema():
            return ten
    raise RuntimeError('Khong tim thay truong nhan. Cac truong co: '
                       + ', '.join(bo.get_field_schema().keys()))

for bo, ten_split in ((train, 'train'), (val, 'val')):
    bo.export(export_dir=THU_MUC,
              dataset_type=fo.types.YOLOv5Dataset,
              label_field=truong_nhan(bo),
              split=ten_split, classes=LOP)

TEP_YAML = os.path.join(THU_MUC, 'dataset.yaml')
assert os.path.isfile(TEP_YAML), 'Xuat du lieu that bai, khong thay dataset.yaml'
print(open(TEP_YAML).read())
print('So anh train:', len(os.listdir(os.path.join(THU_MUC, 'images', 'train'))))
print('So anh val  :', len(os.listdir(os.path.join(THU_MUC, 'images', 'val'))))
print('DA CHUAN BI XONG DU LIEU')


## 4. Huấn luyện

Ô lâu nhất, khoảng 1–2 tiếng. Nó sẽ in ra từng vòng (epoch) một.
Cuối ô phải in ra `DA HUAN LUYEN XONG` kèm đường dẫn tệp `best.pt`.

In [ ]:
import os, glob
from ultralytics import YOLO

TEP_YAML = '/content/noi_that/dataset.yaml'
assert os.path.isfile(TEP_YAML), 'Chua co du lieu. Hay chay lai o so 3.'

model = YOLO('yolov8m.pt')
model.train(
    data=TEP_YAML,
    epochs=60, imgsz=640, batch=16, patience=15,
    degrees=10, translate=0.1, scale=0.5, fliplr=0.5, mosaic=1.0,
    project='/content/runs', name='noi_that',
)

cac_tep = sorted(glob.glob('/content/runs/**/weights/best.pt', recursive=True),
                 key=os.path.getmtime)
assert cac_tep, 'Huan luyen khong sinh ra best.pt — xem lai chu do phia tren.'
BEST = cac_tep[-1]
print('DA HUAN LUYEN XONG')
print('Tep mo hinh:', BEST)


## 5. Đo độ chính xác

Chép bốn con số này vào báo cáo. **mAP@50 trên 0,70 là khá, trên 0,85 là tốt.**

In [ ]:
import os, glob
from ultralytics import YOLO

cac_tep = sorted(glob.glob('/content/runs/**/weights/best.pt', recursive=True),
                 key=os.path.getmtime)
if not cac_tep:
    raise SystemExit('Chua co tep best.pt. O so 4 chua chay xong hoac da bi loi — '
                     'hay chay lai o so 4 va doi den khi hien DA HUAN LUYEN XONG.')
BEST = cac_tep[-1]
print('Dang do mo hinh:', BEST)

kq = YOLO(BEST).val(data='/content/noi_that/dataset.yaml')
print()
print('Precision   :', round(float(kq.box.mp), 4))
print('Recall      :', round(float(kq.box.mr), 4))
print('mAP@50      :', round(float(kq.box.map50), 4))
print('mAP@50-95   :', round(float(kq.box.map), 4))


## 6. Tải mô hình về máy

Chép tệp `best.pt` vừa tải vào thư mục **`models`** của phần mềm rồi chạy lại `chay.bat`.
Phần mềm tự nhận ra và dùng, không phải sửa dòng mã nào.

In [ ]:
import os, glob
from google.colab import files

cac_tep = sorted(glob.glob('/content/runs/**/weights/best.pt', recursive=True),
                 key=os.path.getmtime)
if not cac_tep:
    raise SystemExit('Chua co tep best.pt — hay chay o so 4 truoc.')
files.download(cac_tep[-1])
